In [1]:
%pip install requests --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import subprocess, json, time, requests, pathlib

###############################################################################
# Configuration — only resource group needed
###############################################################################
RESOURCE_GROUP_NAME = "vk-cwyd-data4"  # ← Replace with your resource group

###############################################################################
# Helper — run Azure CLI commands
###############################################################################
def az(cmd: str):
    """Run an Azure CLI command and return parsed JSON (or raw text)."""
    result = subprocess.run(f"az {cmd}", shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"az {cmd}\n{result.stderr.strip()}")
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError:
        return result.stdout.strip()

###############################################################################
# Step 1 — Verify Azure CLI login & discover resources
###############################################################################
account = az("account show")
SUBSCRIPTION_ID = account["id"]
az(f'group show --name "{RESOURCE_GROUP_NAME}"')

search_services = az(f'search service list --resource-group "{RESOURCE_GROUP_NAME}" -o json')
if not search_services:
    raise RuntimeError(f"❌ No Azure Search service found in '{RESOURCE_GROUP_NAME}'.")
AZURE_SEARCH_SERVICE = search_services[0]["name"]

print(f"✅ Search: {AZURE_SEARCH_SERVICE}")

###############################################################################
# Step 1b — Discover App Service & choose output file based on app settings
###############################################################################
web_apps = az(f'webapp list --resource-group "{RESOURCE_GROUP_NAME}" -o json')
if not web_apps:
    raise RuntimeError(f"❌ No App Service found in '{RESOURCE_GROUP_NAME}'.")
APP_SERVICE_NAME = web_apps[0]["name"]
print(f"✅ App Service: {APP_SERVICE_NAME}")

app_settings_list = az(
    f'webapp config appsettings list --name "{APP_SERVICE_NAME}" '
    f'--resource-group "{RESOURCE_GROUP_NAME}" -o json'
)
app_settings = {s["name"]: s["value"] for s in app_settings_list}

USE_INTEGRATED_VECTORIZATION = app_settings.get("AZURE_SEARCH_USE_INTEGRATED_VECTORIZATION", "").strip().lower() == "true"
USE_SEMANTIC_SEARCH          = app_settings.get("AZURE_SEARCH_USE_SEMANTIC_SEARCH", "").strip().lower() == "true"

if USE_INTEGRATED_VECTORIZATION and USE_SEMANTIC_SEARCH:
    OUTPUT_FILE = "azure_search_data_UseVectorization_SemanticSearch.json"
    print("📄 Both AZURE_SEARCH_USE_INTEGRATED_VECTORIZATION and AZURE_SEARCH_USE_SEMANTIC_SEARCH are true.")
    print(f"   → Output file: {OUTPUT_FILE}")
else:
    OUTPUT_FILE = "azure_search_data.json"
    print(f"📄 AZURE_SEARCH_USE_INTEGRATED_VECTORIZATION={app_settings.get('AZURE_SEARCH_USE_INTEGRATED_VECTORIZATION', '<not set>')}")
    print(f"   AZURE_SEARCH_USE_SEMANTIC_SEARCH={app_settings.get('AZURE_SEARCH_USE_SEMANTIC_SEARCH', '<not set>')}")
    print(f"   → Output file: {OUTPUT_FILE}")

###############################################################################
# Step 2 — Enable public access (wrapped in try/finally for cleanup)
###############################################################################
SEARCH_ENABLED = False

try:
    search_public = search_services[0].get("publicNetworkAccess", "enabled").lower()
    if search_public == "disabled":
        print("🔓 Search service public access is disabled — enabling...")
        
        # Get current identity configuration
        identity_type = search_services[0].get("identity", {}).get("type", "None")
        print(f"Current identity type: {identity_type}")
        
        # For UserAssigned identities, use REST API to preserve configuration
        if "UserAssigned" in identity_type:
            print("Using REST API to preserve UserAssigned identity configuration...")
            
            # Get the complete current configuration
            current_config = search_services[0]
            identity_block = current_config.get("identity", {})
            
            # Build PATCH request body
            patch_body = {
                "properties": {
                    "publicNetworkAccess": "enabled"
                },
                "identity": identity_block
            }
            
            # Use az rest to update via Management API
            api_url = (f"https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}/"
                      f"resourceGroups/{RESOURCE_GROUP_NAME}/providers/Microsoft.Search/"
                      f"searchServices/{AZURE_SEARCH_SERVICE}?api-version=2024-03-01-preview")
            
            with open("patch_body.json", "w") as f:
                json.dump(patch_body, f)
            
            az(f'rest --method PATCH --uri "{api_url}" --body @patch_body.json')
            pathlib.Path("patch_body.json").unlink()
            print("✅ Public access enabled for Azure Search service")
        else:
            # For other identity types, use standard CLI
            az(f'search service update --name "{AZURE_SEARCH_SERVICE}" '
               f'--resource-group "{RESOURCE_GROUP_NAME}" --public-access enabled')
            print("✅ Public access enabled for Azure Search service")
        
        SEARCH_ENABLED = True
        
        # Wait and verify access
        print("⏳ Waiting 5 minutes for network changes to propagate...")
        time.sleep(300)  # Initial 5 minute wait
        
        max_additional_wait = 120  # Additional 2 minutes (total 7 min)
        check_interval = 60  # Check every 60 seconds
        elapsed = 0
        
        print("🔍 Verifying Search service public access...")
        while elapsed < max_additional_wait:
            search_status = az(f'search service show --name "{AZURE_SEARCH_SERVICE}" '
                              f'--resource-group "{RESOURCE_GROUP_NAME}"')
            current_access = search_status.get("publicNetworkAccess", "").lower()
            
            if current_access == "enabled":
                print(f"✅ Search service public access confirmed enabled (total wait: {300 + elapsed} seconds)")
                break
            
            print(f"   Still verifying... (checking again in 60s)")
            time.sleep(check_interval)
            elapsed += check_interval
        else:
            print(f"⚠️ Could not confirm Search service access after {300 + max_additional_wait}s total, proceeding anyway...")

    ###########################################################################
    # Step 3 — Get Search admin key & detect index
    ###########################################################################
    AZURE_SEARCH_KEY = az(
        f'search admin-key show --service-name "{AZURE_SEARCH_SERVICE}" '
        f'--resource-group "{RESOURCE_GROUP_NAME}" --query primaryKey -o tsv'
    )

    API_VERSION = "2024-07-01"
    HEADERS = {"Content-Type": "application/json", "api-key": AZURE_SEARCH_KEY}
    
    # Test connectivity first with retry logic
    print("🔍 Testing connectivity to Search service endpoint...")
    connectivity_established = False
    max_connectivity_attempts = 10  # Try for up to 5 minutes
    
    for conn_attempt in range(1, max_connectivity_attempts + 1):
        try:
            resp = requests.get(
                f"https://{AZURE_SEARCH_SERVICE}.search.windows.net/indexes?api-version={API_VERSION}&$select=name",
                headers=HEADERS,
                timeout=30
            )
            if resp.status_code in (200, 404):  # 404 is OK - service is reachable
                print(f"✅ Search service endpoint is accessible (attempt {conn_attempt})")
                connectivity_established = True
                break
        except (requests.exceptions.ConnectTimeout, requests.exceptions.ConnectionError) as e:
            if conn_attempt < max_connectivity_attempts:
                print(f"   Connection attempt {conn_attempt} failed, retrying in 30s...")
                time.sleep(30)
            else:
                raise RuntimeError(f"❌ Cannot connect to Search service endpoint after {max_connectivity_attempts} attempts. "
                                 f"Network changes may need more time to propagate.")
    
    if not connectivity_established:
        raise RuntimeError("❌ Search service endpoint is not accessible")

    # Get the index — prefer index whose name starts with "index"
    resp = requests.get(
        f"https://{AZURE_SEARCH_SERVICE}.search.windows.net/indexes?api-version={API_VERSION}&$select=name",
        headers=HEADERS,
        timeout=30
    )
    if resp.status_code != 200:
        raise RuntimeError(f"❌ Failed to list indexes: {resp.status_code}, {resp.text}")
    indexes = resp.json().get("value", [])
    if not indexes:
        raise RuntimeError("❌ No index found in the search service.")

    # Detect index with "index" prefix
    index_matches = [idx for idx in indexes if idx["name"].lower().startswith("index")]
    if index_matches:
        AZURE_SEARCH_INDEX = index_matches[0]["name"]
    else:
        print("⚠️ No index with prefix 'index' found — falling back to first available index.")
        AZURE_SEARCH_INDEX = indexes[0]["name"]
    print(f"✅ Found index: '{AZURE_SEARCH_INDEX}'")

    ###########################################################################
    # Step 4 — Export all documents (with pagination)
    ###########################################################################
    print("📦 Exporting documents from Search index...")
    ENDPOINT = f"https://{AZURE_SEARCH_SERVICE}.search.windows.net/indexes/{AZURE_SEARCH_INDEX}/docs"
    all_documents = []
    params = {
        "api-version": API_VERSION,
        "search": "*",
        "$top": 1000
    }

    while True:
        response = requests.get(ENDPOINT, headers=HEADERS, params=params, timeout=60)
        if response.status_code != 200:
            raise RuntimeError(f"❌ Failed to fetch data: {response.status_code}, {response.text}")

        data = response.json()
        docs = data.get("value", [])
        all_documents.extend(docs)

        next_link = data.get("@odata.nextLink")
        if not next_link:
            break
        while next_link:
            response = requests.get(next_link, headers=HEADERS, timeout=60)
            if response.status_code != 200:
                raise RuntimeError(f"❌ Failed to fetch next page: {response.status_code}, {response.text}")
            data = response.json()
            all_documents.extend(data.get("value", []))
            next_link = data.get("@odata.nextLink")

    # Save to file
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(all_documents, f, indent=4, ensure_ascii=False)
    
    print(f"✅ Exported {len(all_documents)} documents to {OUTPUT_FILE}")

finally:
    ###########################################################################
    # Step 5 — Restore original access (always runs)
    ###########################################################################
    if SEARCH_ENABLED:
        try:
            print("🔒 Restoring Search service public access to disabled...")
            
            # Get current configuration with identity
            search_svc = az(f'search service show --name "{AZURE_SEARCH_SERVICE}" '
                           f'--resource-group "{RESOURCE_GROUP_NAME}"')
            identity_type = search_svc.get("identity", {}).get("type", "None")
            
            # For UserAssigned identities, use REST API
            if "UserAssigned" in identity_type:
                print("Using REST API to preserve UserAssigned identity configuration...")
                
                identity_block = search_svc.get("identity", {})
                
                # Build PATCH request body
                patch_body = {
                    "properties": {
                        "publicNetworkAccess": "disabled"
                    },
                    "identity": identity_block
                }
                
                # Use az rest to update via Management API
                api_url = (f"https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}/"
                          f"resourceGroups/{RESOURCE_GROUP_NAME}/providers/Microsoft.Search/"
                          f"searchServices/{AZURE_SEARCH_SERVICE}?api-version=2024-03-01-preview")
                
                with open("patch_body.json", "w") as f:
                    json.dump(patch_body, f)
                
                az(f'rest --method PATCH --uri "{api_url}" --body @patch_body.json')
                pathlib.Path("patch_body.json").unlink()
            else:
                # For other identity types, use standard CLI
                az(f'search service update --name "{AZURE_SEARCH_SERVICE}" '
                   f'--resource-group "{RESOURCE_GROUP_NAME}" --public-access disabled')
            
            print("✅ Search service public access restored to disabled.")
        except Exception as e:
            print(f"❌ Search cleanup error: {e}")

    print("\n" + "═" * 63)
    print("  ✅ Export Data — Complete")
    print("═" * 63)
    print(f"  Resource Group        : {RESOURCE_GROUP_NAME}")
    print(f"  Azure Search Service  : {AZURE_SEARCH_SERVICE}")
    print(f"  Azure Search Index    : {AZURE_SEARCH_INDEX}")
    print(f"  Documents exported    : {len(all_documents)}")
    print(f"  Output file           : {OUTPUT_FILE}")
    print("═" * 63)

✅ Search: srch-vkcwyddata4q7pmf
✅ App Service: app-vkcwyddata4q7pmf
📄 Both AZURE_SEARCH_USE_INTEGRATED_VECTORIZATION and AZURE_SEARCH_USE_SEMANTIC_SEARCH are true.
   → Output file: azure_search_data_UseVectorization_SemanticSearch.json
🔍 Testing connectivity to Search service endpoint...
✅ Search service endpoint is accessible (attempt 1)
✅ Found index: 'index-vkcwyddata4q7pmf'
📦 Exporting documents from Search index...
✅ Exported 1000 documents to azure_search_data_UseVectorization_SemanticSearch.json

═══════════════════════════════════════════════════════════════
  ✅ Export Data — Complete
═══════════════════════════════════════════════════════════════
  Resource Group        : vk-cwyd-data4
  Azure Search Service  : srch-vkcwyddata4q7pmf
  Azure Search Index    : index-vkcwyddata4q7pmf
  Documents exported    : 1000
  Output file           : azure_search_data_UseVectorization_SemanticSearch.json
═══════════════════════════════════════════════════════════════
